https://github.com/mihaniky/AI_Architect_3_1_4_Classification

In [12]:
import numpy as np  # быстрые массивы и векторизованная математика (NUMerical PYthon)
import pandas as pd  # табличные данные (DataFrame, Series) (panel data)
import matplotlib.pyplot as plt  # визуализация: диаграммы и графики (MATLAB plotting library)
import seaborn as sns  # статистическая визуализация поверх Matplotlib

from sqlalchemy import create_engine, text # работа напрямую с SQL
# pip install python-dotenv # (Installer Program)
from dotenv import load_dotenv # работа с файлом .env

import os # operating system

# линейные модели: 
# МНК (метод наименьших квадратов) = обычная линейная
# Ridge = L2 (сумма квадратов) = МНК + штраф за большие коэффициенты
# Lasso = L1 (сумма модулей) = МНК + + штраф, обнуляющий часть коэффициентов
from sklearn.linear_model import LinearRegression, Ridge, Lasso  
# масштабирование = стандартизация признаков: (x − mean) / std
from sklearn.preprocessing import StandardScaler  
# разбиение выборки на train/test
from sklearn.model_selection import train_test_split 
# метрики регрессии: MAE, MAPE, MSE, RMSE, R²
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, root_mean_squared_error, r2_score  

### 0. Загрузка данных ###

In [14]:
# текст запроса
query = """
    WITH defect_stats AS (
    SELECT
        p.product_id,
        p.product_name,
        dt.defect_type_id,
        dt.defect_name,
        COUNT(*) AS defect_count
    FROM products p
    JOIN production_orders po     ON po.product_id              = p.product_id
    JOIN quality_inspections qi   ON qi.production_operation_id = po.production_order_id
    JOIN quality_defects qd       ON qd.inspection_id           = qi.inspection_id
    JOIN defect_types dt          ON dt.defect_type_id          = qd.defect_type_id
    GROUP BY p.product_id, p.product_name,
             dt.defect_type_id, dt.defect_name
),
ranked_defect AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY product_id
            ORDER BY defect_count DESC, defect_type_id
        ) AS rn
    FROM defect_stats
),
top_defect AS (
    SELECT
        product_id,
        defect_name,
        defect_count
    FROM ranked_defect
    WHERE rn = 1
),
employee_work AS (
    SELECT
        p.product_id,
        p.product_name,
        e.employee_id,
        e.first_name || ' ' || e.last_name AS employee_name,
        r.role_name,
        -- Считаем часы работы: (ended_at - started_at) в часах
        SUM(
            EXTRACT(EPOCH FROM (
                COALESCE(moa.ended_at, NOW()) - moa.started_at
            )) / 3600.0
        ) AS hours_worked
    FROM products p
    JOIN production_orders po       ON po.product_id           = p.product_id
    JOIN production_operations pop  ON pop.production_order_id = po.production_order_id
    JOIN machine_operator_assignments moa
         ON moa.machine_id          = pop.machine_id
    JOIN employees e                ON e.employee_id           = moa.employee_id
    LEFT JOIN employee_roles r      ON r.role_id               = e.role_id
    GROUP BY p.product_id, p.product_name,
             e.employee_id, employee_name, r.role_name
),
ranked_employee AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY product_id
            ORDER BY hours_worked DESC, employee_id
        ) AS rn
    FROM employee_work
),
top_employee AS (
    SELECT
        product_id,
        employee_name,
        role_name,
        hours_worked
    FROM ranked_employee
    WHERE rn = 1
)
SELECT
    so.sales_order_id,
    so.customer_id,
    soi.product_id,
    p.product_name,
    p.standard_production_time_minutes AS prod_minutes,
    p.standard_cost,
    soi.quantity,
    soi.unit_price,
    pc.category_name,
    COALESCE(td.defect_name, '-') AS top_defect_name,
    COALESCE(te.employee_name, '-') AS top_employee_name,
    COALESCE(td.defect_count, 0) AS top_defect_count,
    COALESCE(ROUND(te.hours_worked::numeric, 2), 0) AS top_employee_hours,
    CASE
        WHEN COALESCE(soi.unit_price, 0) = 0
             OR p.standard_cost * 100.0 / soi.unit_price > 70 THEN 1
        ELSE 0
    END AS is_cost_high
FROM sales_orders so
JOIN sales_order_items soi      ON so.sales_order_id = soi.sales_order_id
JOIN products p                 ON soi.product_id    = p.product_id
LEFT JOIN product_categories pc ON p.category_id     = pc.category_id
LEFT JOIN top_defect td         ON td.product_id     = p.product_id
LEFT JOIN top_employee te       ON te.product_id     = p.product_id
ORDER BY so.order_date DESC, so.sales_order_id, soi.sales_order_item_id;
"""

In [30]:
# явки-пароли (r (= real), как есть, обходит слеши)
ENV_PATH = r"C:\Users\nmm\Documents\GitHub\manufacturing.env"
if os.path.exists(ENV_PATH):
    # load_dotenv(ENV_PATH)
    load_dotenv(ENV_PATH, override=True) # использовать перезапись , т.к. не видит изменений (напр. смена БД)
    # print("База данных:", os.getenv('DB_NAME'))
else:
    print("Файл .env не найден")
host = os.getenv("DB_HOST")
database = os.getenv("DB_NAME")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
# port = os.getenv("DB_PORT")

port = 5432
conn_url = f"postgresql://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(conn_url)



try:
    with engine.connect() as conn:
        df = pd.read_sql(query, conn)
        
except: 
    port = 5433
    conn_url = f"postgresql://{user}:{password}@{host}:{port}/{database}"
    engine = create_engine(conn_url)
    df = pd.read_sql(query, engine)
print(f" Данные получены через порт {port}")


# df.head()

 Данные получены через порт 5432
